## The Conversational "Study Buddy" Bot

- **Objective:** To practice the fundamentals of LangChain's Model I/O, Chains (LCEL), and Memory to build a simple, conversational AI assistant. This assignment does not involve RAG.
- **Scenario:** You will create a "Study Buddy" bot that can explain topics, list key points, and remember your name.
- **Required Libraries:**
  - langchain
  - langchain-google-genai (or langchain-openai)
- **Setup:**
Make sure you have your API key (e.g., GOOGLE_API_KEY) set up as an environment variable or in your notebook's secrets.


**Student Name:** Supriya Misal

In [ ]:
!pip install -q langchain langchain-core langchain-google-genai

In [ ]:
import os
from google.colab import userdata

# Retrieve your Google API key from Colab Secrets
# (Make sure to add 'GOOGLE_API_KEY' in the Secrets tab on the left sidebar)
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.output_parsers import (
    StrOutputParser,
    CommaSeparatedListOutputParser,
)
from langchain_google_genai import ChatGoogleGenerativeAI

# Memory imports: LangChain 1.x deleted the `langchain.memory` module, so
# `ConversationBufferMemory` is gone. The LCEL-native replacement is an
# InMemoryChatMessageHistory store that RunnableWithMessageHistory wraps
# around the chain. See the note above Part 3 for the details.
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
# ==========================================
# PART 1: The Basic Explainer Chain
# ==========================================
print("--- PART 1: Basic Explainer Chain ---")

# 1. Initialize the model
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.3)

# 2. Create the ChatPromptTemplate
explainer_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in one sentence as if to a beginner."
)

# 3. Combine prompt and model using LCEL (|)
explainer_chain = explainer_prompt | llm | StrOutputParser()

# 4. Invoke the chain and print the response
response_part1 = explainer_chain.invoke({"topic": "Quantum Computing"})
print(f"Explanation:\n{response_part1}\n")

In [ ]:
# ==========================================
# PART 2: Adding Structure with an Output Parser
# ==========================================
print("--- PART 2: Structured Output with Parser ---")

# 1. Instantiate the CommaSeparatedListOutputParser
output_parser = CommaSeparatedListOutputParser()

# 2. Create the PromptTemplate with format instructions
list_prompt = PromptTemplate(
    template=(
        "Provide a simple explanation and three key points about {topic}.\n"
        "{format_instructions}"
    ),
    input_variables=["topic"],
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)

# 3. Build the chain: Prompt -> Model -> Output Parser
structured_chain = list_prompt | llm | output_parser

# 4. Invoke and print the parsed Python list
response_part2 = structured_chain.invoke({"topic": "Machine Learning"})
print(f"Parsed List: {response_part2}")
print(f"Output Type: {type(response_part2)}\n")

---

### Note: memory moved in LangChain 1.x

`from langchain.memory import ConversationBufferMemory` fails with
`ModuleNotFoundError: No module named 'langchain.memory'` because LangChain 1.0
removed that module. Two things worth knowing:

- **`langchain_community.memory` is not the fix.** That module exists but is an
  empty namespace and exports nothing, so importing the class from there raises
  `ImportError` instead.
- **`langchain_classic.memory` does still hold it**, so
  `!pip install langchain-classic` plus
  `from langchain_classic.memory import ConversationBufferMemory` is a valid
  one-line escape hatch. But the class is deprecated and warns that it will be
  removed in 2.0.0, so it is not what this notebook uses.

Part 3 below uses the current LCEL-native approach instead:

| Old (LangChain 0.x) | New (LangChain 1.x) |
|---|---|
| `ConversationBufferMemory(return_messages=True)` | `InMemoryChatMessageHistory()` in a per-session store |
| `memory.load_memory_variables({})` before each call | handled by `RunnableWithMessageHistory` |
| `memory.save_context(inputs, outputs)` after each call | handled by `RunnableWithMessageHistory` |
| `memory_key="conversation_history"` | `history_messages_key="conversation_history"` |
| n/a | `config={"configurable": {"session_id": ...}}` selects the conversation |

The `MessagesPlaceholder(variable_name="conversation_history")` in the prompt
stays exactly the same — only how the history reaches it changes.

In [ ]:
# ==========================================
# PART 3: Making it Conversational with Memory
# ==========================================
print("--- PART 3: Conversational Memory Loop ---")

# 1. The session store that replaces ConversationBufferMemory.
#    InMemoryChatMessageHistory holds the running list of BaseMessage objects
#    (the job return_messages=True used to do). Keying it by session_id lets
#    one chain serve several independent conversations.
message_store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Return this session's history, creating it on first use."""
    if session_id not in message_store:
        message_store[session_id] = InMemoryChatMessageHistory()
    return message_store[session_id]


# 2. Create the prompt with a message placeholder for memory
chat_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful and friendly Study Buddy bot. Keep responses concise.",
        ),
        MessagesPlaceholder(variable_name="conversation_history"),
        ("human", "{input}"),
    ]
)

# 3. Build the chain, then wrap it so history is read before every call and
#    both the question and the answer are appended after it. This one wrapper
#    replaces the manual load_memory_variables() / save_context() pairs.
chat_chain = chat_prompt | llm | StrOutputParser()

conversational_chain = RunnableWithMessageHistory(
    chat_chain,
    get_session_history,
    input_messages_key="input",                   # key holding the new user turn
    history_messages_key="conversation_history",  # must match MessagesPlaceholder
)

# The session_id selects which conversation to continue.
config = {"configurable": {"session_id": "study-buddy-alex"}}

# 4. Conversation Turn 1: Introduce name
user_input_1 = "Hi! My name is Alex, and I am studying computer science."
print(f"User: {user_input_1}")

bot_response_1 = conversational_chain.invoke({"input": user_input_1}, config=config)
print(f"Study Buddy: {bot_response_1}\n")

# 5. Conversation Turn 2: Test name recall
#    Nothing is passed in about turn 1 - the wrapper replays it automatically.
user_input_2 = "What is my name and what am I studying?"
print(f"User: {user_input_2}")

bot_response_2 = conversational_chain.invoke({"input": user_input_2}, config=config)
print(f"Study Buddy: {bot_response_2}\n")

# 6. Show what memory actually retained, to prove recall came from history
#    and not from the model guessing.
history = get_session_history("study-buddy-alex").messages
print(f"--- Memory contents: {len(history)} messages ---")
for message in history:
    print(f"  {type(message).__name__:14} {message.content}")